# Projeto — Atividade de Internamento Hospitalar em Portugal

## Etapas do projeto:
1. Preparar os dados (ler CSV → limpar → explorar)
2. Guardar numa base de dados SQL e fazer consultas
3. Analisar e visualizar os resultados

# ESTAÇÃO 1 — Preparar os dados (ler CSV e explorar)

In [19]:


import sqlite3
from pathlib import Path
import pandas as pd

df = pd.read_csv("../data/raw/atividade-de-internamento-hospitalar_fich1.csv",
                  sep=";", encoding="utf-8-sig")

# Renomear colunas para snake_case sem acentos/espaços
df.columns = ["periodo", "regiao","instituicao","localizacao","especialidade","doentes_saidos","dias_internamento"]



print(df.shape)          # (linhas, colunas)
print(df.columns)        # nomes das colunas
display(df.head())         # primeiras 5 linhas
print(df.dtypes)         # tipo de cada coluna (texto, número, etc.)
print(df.isna().sum())   # quantos valores em falta por coluna
display(df.describe().round(2))     # estatísticas básicas das colunas numéricas
print(f"Linhas em bruto: {len(df):,}")# len(df) = nº de linhas.  O  f"...{x:,}"  mostra o número com separador de milhares.



(17616, 7)
Index(['periodo', 'regiao', 'instituicao', 'localizacao', 'especialidade',
       'doentes_saidos', 'dias_internamento'],
      dtype='str')


,periodo,regiao,instituicao,localizacao,especialidade,doentes_saidos,dias_internamento
0,2015-05,Região de Saúde LVT,"Centro Hospitalar de Lisboa Ocidental, EPE","38.708454, -9.216985",Outras Camas,329.0,8327.0
1,2015-05,Região de Saúde Norte,"Centro Hospitalar Trás-os-Montes e Alto Douro,...","41.3031784, -7.7515252",Outras Camas,150.0,2392.0
2,2015-05,Região de Saúde Norte,"Hospital da Senhora da Oliveira, Guimarães, EPE","41.4387173, -8.3086907",Outras Camas,116.0,2066.0
3,2015-05,Região de Saúde Norte,"Hospital de Braga, PPP","41.56785, -8.398982",Outras Camas,71.0,1470.0
4,2015-05,Região de Saúde Norte,"Unidade Local de Saúde de Matosinhos, EPE","41.1794456, -8.6745115",Outras Camas,49.0,802.0


periodo                  str
regiao                   str
instituicao              str
localizacao              str
especialidade            str
doentes_saidos       float64
dias_internamento    float64
dtype: object
periodo               0
regiao                0
instituicao           0
localizacao           0
especialidade         0
doentes_saidos       28
dias_internamento    33
dtype: int64


,doentes_saidos,dias_internamento
count,17588.00,17583.00
mean,3157.16,27088.15
std,3942.83,32915.38
min,0.00,37.00
25%,420.75,4929.00
50%,1710.00,15470.00
75%,4473.25,36047.50
max,34354.00,259313.00


Linhas em bruto: 17,616


# ESTAÇÃO 2 — Guardar numa base de dados e consultar com SQL

O objetivo é guardar os dados de forma estruturada numa base de dados e realizar consultas SQL para explorar a informação.

In [ ]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
con = sqlite3.connect("../data/processed/internamento-hospital.db")


# df.to_sql escreve o DataFrame como uma TABELA dentro da base de dados.
# if_exists='replace' substitui a tabela se já existir; index=False não guarda o nº da linha.

df.to_sql("internamento", con, if_exists="replace", index=False)

q = """

SELECT regiao,        
ROUND(SUM(dias_internamento) / SUM(doentes_saidos)) AS demora_media_regiao
FROM internamento
WHERE doentes_saidos > 0
GROUP BY regiao
ORDER BY demora_media_regiao DESC;

"""
print(pd.read_sql(q, con).to_string(index=False))

q = """
SELECT 
    SUBSTR(periodo, 1, 4) AS ano,
    ROUND(SUM(dias_internamento) / SUM(doentes_saidos)) AS demora_media_anual
FROM internamento
WHERE doentes_saidos > 0
GROUP BY ano
ORDER BY ano;
 """

print(" ")
print("2ª QUERIE compreender a evolução anual da demora média")
print(pd.read_sql(q, con).to_string(index=False))


                     regiao  demora_media_regiao
 Região de Saúde do Algarve                 10.0
        Região de Saúde LVT                  9.0
  Região de Saúde do Centro                  8.0
Região de Saúde do Alentejo                  8.0
      Região de Saúde Norte                  8.0
 
2ª QUERIE compreender a evolução anual da demora média
 ano  demora_media_anual
2015                 8.0
2016                 8.0
2017                 8.0
2018                 9.0
2019                 9.0
2020                 9.0
2021                 9.0
2022                 9.0
2023                 9.0
2024                 9.0
2025                 9.0
2026                 9.0
